# Как мы строили временные признаки для рекомендаций

Это ноутбук с основными экспериментами для отбора в T-Lab. Данные взяты из [KION Dataset](https://github.com/irsafilo/KION_DATASET). Сначала соберём ручные признаки и обучим CatBoost, затем проверим несколько способов учитывать прошлые просмотры: фиксированные окна, последние N событий и экспоненциальное затухание. В конце сравним модели на следующей неделе и оценим разброс результатов.

Для всех моделей списки рекомендуемых фильмов строятся одинаково: выбираем популярные фильмы, исключаем уже просмотренные и ранжируем оставшиеся. История для признаков всегда заканчивается до даты, на которую делается прогноз.

В обучении оставлен тот же способ формирования примеров и ранней остановки, что использовался во время экспериментов. Он не идеален: часть пользователей выбирается по наличию будущих просмотров. Менять его сейчас означало бы заново обучать модели и получать другие результаты. Ниже и в `README.md` это ограничение разобрано отдельно.

**Как запустить:** скачайте исходные `interactions.csv`, `items.csv` и `users.csv` из KION, положите их в папку `data_original` рядом с ноутбуком и выполните все ячейки с начала. Использованные версии библиотек записаны в `requirements.txt`.

Результаты в отчёте взяты из исходного запуска. После подготовки чистовой версии полный повторный запуск на CSV не проводился, поэтому перед отправкой нужно проверить её командой **Restart & Run All**.


## 1. Данные и временное разбиение

Для экспериментов не нужен весь KION: на миллионах пользователей каждый новый вариант признаков считался бы слишком долго. Возьмём случайные 10 000 пользователей, у которых до 19 июля 2021 года было хотя бы три взаимодействия, и сохраним всю доступную историю каждого из них. Так данные останутся достаточно небольшими для экспериментов, хотя выводы уже нельзя автоматически переносить на всю аудиторию.

Модели обучаются на трёх недельных срезах, начинающихся 19 и 26 июля и 2 августа. Следующая неделя, с 9 августа, нужна для выбора моделей; неделя с 16 августа оставлена для итогового теста. Для сравнения заранее выберем 3000 пользователей из нашей подвыборки, не заглядывая в их будущие просмотры.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import catboost
from catboost import CatBoostClassifier

DATA_DIR = Path('data_original')
filenames = ['interactions.csv', 'items.csv', 'users.csv']
missing = [str(DATA_DIR / name) for name in filenames if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError('Не найдены CSV KION: ' + ', '.join(missing) + '. См. README.md')

df_interactions = pd.read_csv(DATA_DIR / 'interactions.csv')
df_items = pd.read_csv(DATA_DIR / 'items.csv')
df_users = pd.read_csv(DATA_DIR / 'users.csv')
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'])
print('Записей о просмотрах:', len(df_interactions), 'Фильмов:', df_items.item_id.nunique())

In [ ]:
cutoff = pd.Timestamp("2021-07-19")

history = df_interactions[
    df_interactions["last_watch_dt"] < cutoff
]

user_counts = history.groupby("user_id").size()

eligible_users = user_counts[
    user_counts >= 3
].index

sampled_users = pd.Series(eligible_users).sample(
    n=min(10000, len(eligible_users)),
    random_state=42
)

df_small = df_interactions[
    df_interactions["user_id"].isin(sampled_users)
].copy()

print(df_small.shape)
print(df_small["user_id"].nunique())

In [ ]:
train_cutoffs = ['2021-07-19', '2021-07-26', '2021-08-02']
val_cutoff = pd.Timestamp('2021-08-09')
test_cutoff = pd.Timestamp('2021-08-16')
assert pd.Timestamp(train_cutoffs[-1]) + pd.Timedelta(days=7) == val_cutoff
assert val_cutoff + pd.Timedelta(days=7) == test_cutoff

# Список пользователей фиксируем заранее: их будущие просмотры на выбор не влияют.
eval_users = (df_small.loc[
    df_small['last_watch_dt'] < '2021-07-19', 'user_id'
].drop_duplicates().sample(n=3000, random_state=42).tolist())
print('Пользователей для проверки:', len(eval_users))

## 2. Кандидаты, метки и признаки

В каждый момент прогноза берём 1000 самых популярных фильмов по уже известным просмотрам. Из них убираем всё, что пользователь раньше смотрел, и оставляем первые 200. Это общий набор кандидатов для всех методов.

Метка **1** означает, что в следующие семь дней пользователь посмотрел не меньше 50% фильма. Если такого события не было, ставим **0**. При этом ноль не обязательно означает, что фильм пользователю не понравился: у нас нет данных о том, видел ли он рекомендацию вообще.


Функции ниже строят пары «пользователь — фильм», но для обучения и для проверки делают это по-разному.

`build_pairs` оставляет пользователей, у которых в следующую неделю был положительный просмотр. На обучении дополнительно берём до 20 отрицательных кандидатов на каждый положительный, чтобы не раздувать таблицу. Этот отбор использует будущие события и делает обучающую выборку смещённой. Сама история, из которой считаются признаки, при этом остаётся строго прошлой.

`build_eval_pairs` устроена иначе: она строит кандидатов для заранее выбранных 3000 пользователей независимо от того, что они посмотрят позже. Именно эту выборку будем использовать для сравнения качества ранжирования.


In [ ]:
def build_pairs(df, cutoff, k=200, training=False):
    cutoff = pd.Timestamp(cutoff)

    history = df[df["last_watch_dt"] < cutoff]
    future = df[
        (df["last_watch_dt"] >= cutoff) &
        (df["last_watch_dt"] < cutoff + pd.Timedelta(days=7)) &
        (df["watched_pct"] >= 50)
    ]

    # Популярность рассчитываем только по событиям до даты прогноза.
    popular = history["item_id"].value_counts().head(1000).index.tolist()

    seen = history.groupby("user_id")["item_id"].agg(set).to_dict()
    targets = future.groupby("user_id")["item_id"].agg(set).to_dict()

    rng = np.random.default_rng(42)
    rows = []

    for user, truth in targets.items():
        if user not in seen:
            continue

        truth = truth - seen[user]
        if not truth:
            continue

        candidates = [
            item for item in popular
            if item not in seen[user]
        ][:k]

        if training:
            # На обучении оставляем позитивные пары и ограничиваем число остальных.
            positives = [i for i in candidates if i in truth]
            if not positives:
                continue

            negatives = [i for i in candidates if i not in truth]
            n_neg = min(len(negatives), 20 * len(positives))

            negatives = rng.choice(
                negatives, size=n_neg, replace=False
            ).tolist()

            candidates = positives + negatives

        rows.extend(
            (user, item, int(item in truth), cutoff)
            for item in candidates
        )

    return pd.DataFrame(
        rows,
        columns=["user_id", "item_id", "label", "cutoff"]
    )

In [ ]:
def build_eval_pairs(df, cutoff, users, k=200):
    cutoff = pd.Timestamp(cutoff)

    history = df[df["last_watch_dt"] < cutoff]

    future = df[
        (df["last_watch_dt"] >= cutoff) &
        (df["last_watch_dt"] < cutoff + pd.Timedelta(days=7)) &
        (df["watched_pct"] >= 50)
    ]

    popular = history["item_id"].value_counts().head(1000).index.tolist()
    seen = history.groupby("user_id")["item_id"].agg(set).to_dict()
    targets = future.groupby("user_id")["item_id"].agg(set).to_dict()

    rows = []

    # Здесь проходим по заранее выбранным пользователям, а не только по будущим активным.
    for user in users:
        if user not in seen:
            continue

        relevant = targets.get(user, set()) - seen[user]

        candidates = [
            item for item in popular
            if item not in seen[user]
        ][:k]

        rows.extend(
            (user, item, int(item in relevant), cutoff)
            for item in candidates
        )

    return pd.DataFrame(
        rows,
        columns=["user_id", "item_id", "label", "cutoff"]
    )

In [ ]:
df_train = pd.concat(
    [build_pairs(df_small, t, training=True) for t in train_cutoffs],
    ignore_index=True
)
# Отдельная выборка для ранней остановки: её состав зависит от будущих просмотров.
df_val_es = build_pairs(df_small, val_cutoff, training=False)
# Для сравнения ранжирования используем заранее выбранных людей. NDCG всё равно условный.
df_val_fixed = build_eval_pairs(df_small, val_cutoff, eval_users, k=200)
assert set(df_val_fixed['user_id']) == set(eval_users)
print('Обучающих пар:', len(df_train), 'Пар для ранней остановки:', len(df_val_es),
      'Пар для оценки качества:', len(df_val_fixed))

In [ ]:
def add_features(pairs):
    result = []

    item_meta = df_items[
        ["item_id", "release_year", "age_rating", "content_type", "genres"]
    ].copy()

    # Пока работаем с первым жанром фильма, остальные жанры не учитываем.
    item_meta["main_genre"] = (
        item_meta["genres"].fillna("unknown")
        .str.split(",").str[0].str.strip()
    )

    for cutoff, part in pairs.groupby("cutoff"):
        # На каждом срезе признаки видят только уже произошедшие просмотры.
        history = df_small[df_small["last_watch_dt"] < cutoff]
        f = part.copy()

        for days in [7, 30]:
            h = history[
                history["last_watch_dt"] >= cutoff - pd.Timedelta(days=days)
            ]

            for key, prefix in [
                ("user_id", "user"),
                ("item_id", "item")
            ]:
                counts = h.groupby(key).size()
                f[f"{prefix}_count_{days}d"] = (
                    f[key].map(counts).fillna(0)
                )

        last_watch = history.groupby("user_id")["last_watch_dt"].max()
        f["user_recency"] = (
            cutoff - f["user_id"].map(last_watch)
        ).dt.days

        f = f.merge(
            item_meta.drop(columns="genres"),
            on="item_id",
            how="left"
        )

        h30 = history[
            history["last_watch_dt"] >= cutoff - pd.Timedelta(days=30)
        ].merge(
            item_meta[["item_id", "main_genre"]],
            on="item_id",
            how="left"
        )

        genre_counts = (
            h30.groupby(["user_id", "main_genre"])
            .size()
            .rename("user_genre_count_30d")
            .reset_index()
        )

        f = f.merge(
            genre_counts,
            on=["user_id", "main_genre"],
            how="left"
        )

        f["user_genre_count_30d"] = (
            f["user_genre_count_30d"].fillna(0)
        )

        f["user_genre_share_30d"] = (
            f["user_genre_count_30d"] /
            (f["user_count_30d"] + 1)
        )

        f = f.merge(
            df_users,
            on="user_id",
            how="left"
        )

        for c in ["content_type", "main_genre", "age", "income", "sex"]:
            f[c] = f[c].fillna("unknown").astype(str)

        result.append(f)

    return pd.concat(result, ignore_index=True)

In [ ]:
genre_map = (
    df_items[['item_id', 'genres']].drop_duplicates('item_id')
    .assign(main_genre=lambda d: d['genres'].fillna('unknown')
            .str.split(',').str[0].str.strip())
    .set_index('item_id')['main_genre']
)

In [ ]:
def generate_auto_features(pairs, windows=(3, 7, 14, 30)):
    result = []

    for cutoff, part in pairs.groupby("cutoff"):
        h = df_small.loc[
            df_small["last_watch_dt"] < cutoff,
            ["user_id", "item_id", "last_watch_dt", "watched_pct"]
        ].copy()

        h["main_genre"] = h["item_id"].map(genre_map).fillna("unknown")

        f = part.copy()
        if "main_genre" not in f.columns:
            f["main_genre"] = f["item_id"].map(genre_map).fillna("unknown")

        # Для каждого окна считаем число событий и средний процент просмотра.
        for w in windows:
            hw = h[
                h["last_watch_dt"] >= cutoff - pd.Timedelta(days=w)
            ]

            groups = [
                (["user_id"], "user", f["user_id"]),
                (["item_id"], "item", f["item_id"]),
                (
                    ["user_id", "main_genre"],
                    "user_genre",
                    pd.MultiIndex.from_frame(f[["user_id", "main_genre"]])
                )
            ]

            for keys, name, idx in groups:
                grouped = hw.groupby(keys)["watched_pct"]

                f[f"auto_{name}_count_{w}d"] = (
                    grouped.size()
                    .reindex(idx, fill_value=0)
                    .to_numpy(dtype="float32")
                )

                f[f"auto_{name}_mean_pct_{w}d"] = (
                    grouped.mean()
                    .reindex(idx)
                    .fillna(0)
                    .to_numpy(dtype="float32")
                )

        result.append(f)

    return pd.concat(result, ignore_index=True)

In [ ]:
def add_history_features(pairs):
    result = []

    for cutoff, part in pairs.groupby("cutoff"):
        # Эти признаки берутся за всю прошлую историю, а не за последние 30 дней.
        h = df_small[df_small["last_watch_dt"] < cutoff]
        f = part.copy()

        for key, name in [
            ("user_id", "user"),
            ("item_id", "item")
        ]:
            counts = h.groupby(key).size()
            means = h.groupby(key)["watched_pct"].mean()

            f[f"hist_{name}_count"] = (
                f[key].map(counts).fillna(0)
            )

            f[f"hist_{name}_mean_pct"] = (
                f[key].map(means).fillna(0)
            )

        result.append(f)

    return pd.concat(result, ignore_index=True)

In [ ]:
def add_last_n_features(pairs, Ns=(3, 7, 14, 30)):
    result = []

    for cutoff, part in pairs.groupby("cutoff"):
        h = df_small[
            df_small["last_watch_dt"] < cutoff
        ][[
            "user_id", "item_id",
            "last_watch_dt", "watched_pct"
        ]].sort_values("last_watch_dt").copy()

        h["main_genre"] = (
            h["item_id"].map(genre_map).fillna("unknown")
        )

        f = part.copy()

        if "main_genre" not in f.columns:
            f["main_genre"] = (
                f["item_id"].map(genre_map).fillna("unknown")
            )

        ug_idx = pd.MultiIndex.from_frame(
            f[["user_id", "main_genre"]]
        )

        for n in Ns:
            hu = h.groupby("user_id", sort=False).tail(n)
            hi = h.groupby("item_id", sort=False).tail(n)

            # Средний процент просмотра: отдельно для пользователя и для фильма.
            u_mean = hu.groupby("user_id")["watched_pct"].mean()
            i_mean = hi.groupby("item_id")["watched_pct"].mean()

            f[f"last{n}_user_mean_pct"] = (
                f["user_id"].map(u_mean).fillna(0).to_numpy()
            )

            f[f"last{n}_item_mean_pct"] = (
                f["item_id"].map(i_mean).fillna(0).to_numpy()
            )

            # Как часто пользователь смотрел этот жанр среди последних N своих событий.
            genre_counts = hu.groupby(
                ["user_id", "main_genre"]
            ).size()

            user_counts = hu.groupby("user_id").size()

            numerator = genre_counts.reindex(
                ug_idx, fill_value=0
            ).to_numpy()

            denominator = (
                f["user_id"].map(user_counts)
                .fillna(0).clip(lower=1).to_numpy()
            )

            f[f"last{n}_genre_share"] = (
                numerator / denominator
            )

            # Сколько дней прошло между первым и последним из N просмотров фильма.
            item_span = (
                hi.groupby("item_id")["last_watch_dt"].max()
                - hi.groupby("item_id")["last_watch_dt"].min()
            ).dt.total_seconds() / 86400

            f[f"last{n}_item_span"] = (
                f["item_id"].map(item_span).fillna(0).to_numpy()
            )

        result.append(f)

    return pd.concat(result, ignore_index=True)

In [ ]:
def add_decay_features(pairs, taus=(3, 14, 30)):
    result = []

    for cutoff, part in pairs.groupby("cutoff"):
        h = df_small[
            df_small["last_watch_dt"] < cutoff
        ].copy()

        h["main_genre"] = (
            h["item_id"].map(genre_map).fillna("unknown")
        )

        f = part.copy()

        if "main_genre" not in f:
            f["main_genre"] = (
                f["item_id"].map(genre_map).fillna("unknown")
            )

        age = (
            (cutoff - h["last_watch_dt"])
            .dt.total_seconds() / 86400
        )

        # Старые просмотры получают меньший вес; скорость снижения задаёт tau.
        for tau in taus:
            h["_weight"] = np.exp(-age / tau)

            item_score = h.groupby("item_id")["_weight"].sum()

            f[f"decay_item_{tau}"] = (
                f["item_id"].map(item_score).fillna(0)
            )

            ug_score = (
                h.groupby(["user_id", "main_genre"])["_weight"]
                .sum()
                .rename(f"decay_user_genre_{tau}")
                .reset_index()
            )

            f = f.merge(
                ug_score,
                on=["user_id", "main_genre"],
                how="left"
            )

            col = f"decay_user_genre_{tau}"
            f[col] = f[col].fillna(0)

        result.append(f)

    return pd.concat(result, ignore_index=True)

## 3. Какие признаки сравниваем

Сначала собираем **Manual+**. В нём 19 признаков: сведения о пользователе и фильме, давность просмотра, число взаимодействий за 7 и 30 дней, интерес к основному жанру, а также статистики за всю доступную прошлую историю. Для жанров здесь берётся только первый жанр из списка. Это упрощение, о котором стоит помнить при оценке качества.

К этой общей основе по очереди добавляем разные способы описать историю:

- **Fixed Windows.** Считаем количество просмотров и средний `watched_pct` за 3, 7, 14 и 30 дней для пользователя, фильма и пары «пользователь — жанр». После исключения совпадений с ручными признаками остаётся 19 новых столбцов.
- **Last-N.** Берём последние 3, 7, 14 или 30 взаимодействий, не привязываясь к календарным дням. Например, считаем, какая доля недавних просмотров пользователя относится к жанру кандидата. Всего 16 новых признаков.
- **Exponential.** Старые события не отбрасываются резко после определённого дня. Вместо этого каждому событию задаём вес `exp(-age / τ)`: чем оно старше, тем меньше влияет. Используем `τ = 3, 14, 30` дней, отдельно для популярности фильма и интереса пользователя к жанру. Получается 6 новых признаков.
- **Adaptive Memory.** Оставляем только по одному экспоненциальному признаку для фильма и для жанра. Какие именно `τ` взять, определяем по важности признаков в предварительном CatBoost.

Это не нейросеть, которая обучает собственную память для каждого пользователя. Последний вариант просто автоматически выбирает два признака из шести подготовленных.


In [ ]:
features = [
    'user_count_7d', 'user_count_30d', 'item_count_7d', 'item_count_30d',
    'user_recency', 'user_genre_count_30d', 'user_genre_share_30d',
    'release_year', 'age_rating', 'content_type', 'main_genre',
    'age', 'income', 'sex', 'kids_flg'
]
cat_features = ['content_type', 'main_genre', 'age', 'income', 'sex']
manual_extra = ['hist_user_count', 'hist_user_mean_pct',
                'hist_item_count', 'hist_item_mean_pct']
base_cols = features + manual_extra
existing = {
    'auto_user_count_7d', 'auto_user_count_30d',
    'auto_item_count_7d', 'auto_item_count_30d',
    'auto_user_genre_count_30d'
}

def build_all_features(pairs):
    # Считаем всё один раз; ниже каждой модели передадим только нужные столбцы.
    x = add_features(pairs)
    x = generate_auto_features(x)
    x = add_history_features(x)
    x = add_last_n_features(x)
    x = add_decay_features(x)
    assert len(x) == len(pairs), 'При объединении признаков изменилось число строк'
    return x

train_af = build_all_features(df_train)
val_af = build_all_features(df_val_es)
val_fixed = build_all_features(df_val_fixed)

auto_cols = [c for c in train_af if c.startswith('auto_') and c not in existing]
decay_cols = [f'decay_{ent}_{tau}' for ent in ['item', 'user_genre']
              for tau in [3, 14, 30]]
lastn_cols = [f'last{n}_{name}' for n in [3, 7, 14, 30]
              for name in ['user_mean_pct','item_mean_pct','genre_share','item_span']]
assert (len(features),len(base_cols),len(auto_cols),len(decay_cols),len(lastn_cols)) == (15,19,19,6,16)
assert len(train_af) == 50422
print('Признаков: ручных',len(features),'Manual+',len(base_cols),'AutoFE',len(auto_cols),
      'Exponential',len(decay_cols),'Last-N',len(lastn_cols))

In [ ]:
# Предварительная модель помогает выбрать tau отдельно для фильма и жанра.
probe = CatBoostClassifier(iterations=180, depth=6, learning_rate=0.05,
                            random_seed=42, verbose=False)
probe_cols = base_cols + decay_cols
probe.fit(train_af[probe_cols], train_af['label'], cat_features=cat_features)
imp = pd.Series(probe.get_feature_importance(), index=probe_cols)
adaptive_cols = []
for entity in ['item','user_genre']:
    candidates = [f'decay_{entity}_{tau}' for tau in [3,14,30]]
    adaptive_cols.append(imp[candidates].idxmax())
print('Для адаптивной памяти выбраны:',adaptive_cols)

## 4. Обучаем модели и смотрим на валидацию

Для всех вариантов используем CatBoost с одинаковыми настройками. Число деревьев выбирается с помощью ранней остановки на `df_val_es`. Эта выборка была собрана по будущим положительным просмотрам, поэтому её распределение отличается от обычных рекомендательных списков. Качество ранжирования отдельно считаем на `val_fixed`, где пользователи выбраны заранее.

Основная метрика, **NDCG@10**, проверяет, насколько высоко оказались фильмы, которые пользователь действительно посмотрел на следующей неделе. При расчёте идеального результата учитываются *все* новые положительные просмотры, даже если нужный фильм не попал в наши 200 кандидатов.

Если за неделю пользователь не посмотрел ни одного нового фильма на 50% и больше, его NDCG не считаем. Поэтому среднее ниже относится только к пользователям с такими будущими просмотрами, а не ко всем 3000.


In [ ]:
def evaluate_ndcg(pairs, score_col, cutoff, k=10):
    cutoff = pd.Timestamp(cutoff)
    history = df_small[df_small['last_watch_dt'] < cutoff]
    future = df_small[
        (df_small['last_watch_dt'] >= cutoff) &
        (df_small['last_watch_dt'] < cutoff + pd.Timedelta(days=7)) &
        (df_small['watched_pct'] >= 50)
    ]
    seen = history.groupby('user_id')['item_id'].agg(set).to_dict()
    truth = future.groupby('user_id')['item_id'].agg(set).to_dict()
    results = []
    for user, group in pairs.groupby('user_id'):
        relevant = truth.get(user, set()) - seen.get(user, set())
        if not relevant:  # Без положительных событий за неделю NDCG не определяем.
            continue
        top_items = group.nlargest(k, score_col)['item_id']
        dcg = sum((item in relevant) / np.log2(rank + 2)
                  for rank, item in enumerate(top_items))
        # В идеальный DCG входят все подходящие фильмы, даже отсутствующие среди кандидатов.
        idcg = sum(1 / np.log2(rank + 2)
                   for rank in range(min(len(relevant), k)))
        results.append(dcg / idcg)
    return float(np.mean(results)) if results else float('nan')

In [ ]:
variants = {
    'Manual+': base_cols,
    'Fixed Windows': base_cols + auto_cols,
    'Last-N': base_cols + lastn_cols,
    'Exponential': base_cols + decay_cols,
    'Adaptive Memory': base_cols + adaptive_cols
}
trained_models = {}
results = []
for name, cols in variants.items():
    m = CatBoostClassifier(iterations=600, depth=6, learning_rate=0.05,
                           loss_function='Logloss', random_seed=42,
                           verbose=False)
    m.fit(train_af[cols], train_af['label'], cat_features=cat_features,
          eval_set=(val_af[cols], val_af['label']), early_stopping_rounds=60)
    val_fixed[f'pred_{name}'] = m.predict_proba(val_fixed[cols])[:,1]
    results.append({'method':name, 'ndcg':evaluate_ndcg(val_fixed,f'pred_{name}',val_cutoff),
                    'num_features':len(cols), 'trees':m.tree_count_})
    trained_models[name] = m
val_results = pd.DataFrame(results)
print(val_results.sort_values('ndcg',ascending=False).to_string(index=False))

Для полноты оставим ещё результат самого первого AutoFE: к первоначальным 15 ручным признакам добавлялись 19 автоматически построенных. Позже ручную основу усилили четырьмя признаками за всю историю. Поэтому **старый All AutoFE и новый Manual+ нельзя напрямую сравнивать как две версии одной модели, отличающиеся только AutoFE**. Для такого сравнения выше уже есть отдельный вариант `Fixed Windows` с общей основой.


In [ ]:
all_features = features + auto_cols
model_auto = CatBoostClassifier(iterations=600, depth=6, learning_rate=0.05,
                                loss_function='Logloss', eval_metric='Logloss',
                                random_seed=42, verbose=False)
model_auto.fit(train_af[all_features], train_af['label'], cat_features=cat_features,
               eval_set=(val_af[all_features], val_af['label']), early_stopping_rounds=60)
val_fixed['pred_All AutoFE'] = model_auto.predict_proba(val_fixed[all_features])[:,1]
print('Первоначальный All AutoFE на валидации:',evaluate_ndcg(val_fixed,'pred_All AutoFE',val_cutoff))

## 5. Зависит ли лучший способ памяти от активности пользователя?

Пользователи отличаются по количеству просмотров. У кого-то за неделю накапливается длинная история, а кто-то смотрит фильмы редко. Проверим, меняется ли качество методов между тремя группами: 1–5, 6–15 и 16+ взаимодействий до 9 августа.

По результатам валидации в каждой группе выбираем модель с самым высоким NDCG и получаем простое правило: для первой группы `Adaptive Memory`, для второй `Exponential`, для третьей `Last-N`. Ниже оно называется **Activity-Aware Routing**.

Здесь есть подвох: лучшие модели для групп выбраны по той же валидации, на которой мы оцениваем итоговое правило. Значит, получившееся значение NDCG может быть завышено. Проверять, сохранился ли выигрыш, будем только на следующей неделе.


In [ ]:
hist_counts = (
    df_small.loc[df_small['last_watch_dt'] < val_cutoff]
    .groupby('user_id').size()
)
val_fixed['activity_group'] = pd.cut(
    val_fixed['user_id'].map(hist_counts), bins=[0,5,15,np.inf],
    labels=['1–5','6–15','16+']
)
for activity, sub in val_fixed.groupby('activity_group',observed=True):
    print('\nГруппа:',activity,'всего пользователей:',sub.user_id.nunique())
    for name in variants:
        print(name,round(evaluate_ndcg(sub,f'pred_{name}',val_cutoff),5))

# Подбор схемы сделан на этой же валидации: её итоговая оценка завышена.
routing = {'1–5':'Adaptive Memory','6–15':'Exponential','16+':'Last-N'}
def apply_routing(df):
    return np.select(
        [df['activity_group'].astype(str).eq(g) for g in routing],
        [df[f'pred_{name}'] for name in routing.values()],
        default=df['pred_Manual+']
    )
val_fixed['score_routed'] = apply_routing(val_fixed)
print('Маршрутизация на валидации:',evaluate_ndcg(val_fixed,'score_routed',val_cutoff))

## 6. Проверка на отложенной неделе: 16–22 августа

К этому моменту признаки, модели и правило выбора по активности уже зафиксированы по обучению и валидации. Теперь проверяем их на следующей неделе, которую не использовали для выбора параметров.

Списки кандидатов строятся тем же способом, но вся история для них и для признаков обрывается **до 16 августа**. После просмотра этих результатов ничего в настройках не меняем, иначе тест перестанет быть независимым.


In [ ]:
df_test_fixed = build_eval_pairs(df_small,test_cutoff,eval_users,k=200)
test_fixed = build_all_features(df_test_fixed)
counts_test = (
    df_small.loc[df_small['last_watch_dt'] < test_cutoff]
    .groupby('user_id').size()
)
test_fixed['activity_group'] = pd.cut(
    test_fixed['user_id'].map(counts_test), bins=[0,5,15,np.inf],
    labels=['1–5','6–15','16+']
)
for name, cols in variants.items():
    test_fixed[f'pred_{name}'] = trained_models[name].predict_proba(test_fixed[cols])[:,1]
test_fixed['pred_All AutoFE'] = model_auto.predict_proba(test_fixed[all_features])[:,1]
test_fixed['score_routed'] = apply_routing(test_fixed)
report_cols = {
    'Manual+':'pred_Manual+',
    'Fixed Windows':'pred_Fixed Windows',
    'All AutoFE (original base)':'pred_All AutoFE',
    'Exponential':'pred_Exponential',
    'Activity-Aware':'score_routed'
}
for name,col in report_cols.items():
    print(name,':',evaluate_ndcg(test_fixed,col,test_cutoff))

## 7. Насколько надёжна разница в NDCG?

Среднее значение может заметно зависеть от нескольких пользователей. Поэтому сначала посчитаем NDCG@10 отдельно для каждого человека, а затем сравним модели с помощью **парного bootstrap**.

Мы 2000 раз случайно набираем пользователей с возвращением. В каждом повторе для обеих моделей берём одних и тех же людей и считаем среднюю разницу NDCG. По получившимся разницам строим 95%-й доверительный интервал.

Этот расчёт показывает неопределённость внутри одной тестовой недели. Он не отвечает на вопрос, что произойдёт на другой неделе или при повторном обучении. В исходном тесте в среднее NDCG вошли лишь 254 пользователя из 3000: у остальных не было нового положительного просмотра.


In [ ]:
history_test = df_small[df_small['last_watch_dt'] < test_cutoff]
future_test = df_small[
    (df_small['last_watch_dt'] >= test_cutoff) &
    (df_small['last_watch_dt'] < test_cutoff + pd.Timedelta(days=7)) &
    (df_small['watched_pct'] >= 50)
]
seen_test = history_test.groupby('user_id')['item_id'].agg(set).to_dict()
truth_test = future_test.groupby('user_id')['item_id'].agg(set).to_dict()
columns = {'Manual+':'pred_Manual+', 'Exponential':'pred_Exponential',
           'Activity-Aware':'score_routed'}
rows=[]
for user,group in test_fixed.groupby('user_id'):
    relevant=truth_test.get(user,set())-seen_test.get(user,set())
    if not relevant:
        continue
    idcg=sum(1/np.log2(i+2) for i in range(min(len(relevant),10)))
    row={'user_id':user}
    for name,col in columns.items():
        top=group.nlargest(10,col)['item_id']
        dcg=sum((item in relevant)/np.log2(i+2) for i,item in enumerate(top))
        row[name]=dcg/idcg
    rows.append(row)
user_metrics=pd.DataFrame(rows)
print('Пользователей, вошедших в NDCG:',len(user_metrics))
print(user_metrics[list(columns)].mean())
rng=np.random.default_rng(42)
B=2000
indices=rng.integers(0,len(user_metrics),size=(B,len(user_metrics)))
for a,b in [('Exponential','Manual+'),('Activity-Aware','Exponential')]:
    diff=(user_metrics[a]-user_metrics[b]).to_numpy()
    bs=diff[indices].mean(axis=1)
    ci=np.quantile(bs,[0.025,0.975])
    print(f'{a} - {b}: delta={diff.mean():.6f}; CI=[{ci[0]:.6f},{ci[1]:.6f}]')

## 8. Время расчёта и память

В качестве небольшой проверки затрат возьмём 600 000 пар из фиксированной валидационной выборки. Замерим отдельно построение ручных признаков, дополнительных окон и экспоненциальной памяти. Затем посмотрим, сколько времени CatBoost тратит на предсказания и какой объём занимают выбранные столбцы в pandas.

Это одиночные замеры на одном компьютере, а не полноценный тест производительности. Они позволяют сравнить наши варианты в текущей реализации, но не показывают пиковое потребление памяти или скорость обучения на больших данных.


In [ ]:
from time import perf_counter
raw = df_val_fixed
t0 = perf_counter(); manual_bench=add_features(raw); t_manual=perf_counter()-t0
t0 = perf_counter(); base_bench=add_history_features(manual_bench); t_history=perf_counter()-t0
t0 = perf_counter(); fixed_bench=generate_auto_features(base_bench); t_fixed=perf_counter()-t0
t0 = perf_counter(); exp_bench=add_decay_features(base_bench); t_exp=perf_counter()-t0
print('Кандидатных пар:',len(raw),'ручные признаки (с):',round(t_manual,3),
      'вся история (с):',round(t_history,3),
      'доп. окна (с):',round(t_fixed,3),'затухание (с):',round(t_exp,3))
for name,df in [('Manual+',base_bench),('Fixed Windows',fixed_bench),('Exponential',exp_bench)]:
    cols=variants[name]
    t0=perf_counter();_=trained_models[name].predict_proba(df[cols])[:,1];t_predict=perf_counter()-t0
    memory_mb=df[cols].memory_usage(deep=True).sum()/2**20
    print(name,len(cols),'признаков;',round(t_predict,3),'с на предсказание;',
          round(memory_mb,1),'МиБ под столбцы признаков')

## 9. Что ограничивает результаты и как повторить эксперимент

У этого эксперимента есть несколько ограничений, которые важно учитывать перед тем, как делать выводы.

**Выборка.** Мы взяли 10 000 пользователей, у которых до 19 июля было хотя бы три взаимодействия. Это позволило быстрее проверять идеи, но такие пользователи могут быть активнее среднестатистического зрителя KION. Даже популярность фильмов мы считали только по их просмотрам.

**Обучающие примеры.** Положительной меткой считается просмотр хотя бы половины фильма в течение следующей недели. Для остальных кандидатов ставится ноль, хотя мы не знаем, показывали ли человеку этот фильм. Кроме того, на обучении и при выборе числа деревьев состав пользователей зависел от будущих положительных просмотров. Из-за этого результаты могут отличаться от качества на обычном потоке пользователей. При расчёте самих признаков будущие события не использовались.

**Небольшое число пользователей в тесте.** Мы заранее выбрали 3000 человек, но NDCG усредняется только по тем, кто действительно посмотрел новый фильм хотя бы наполовину. На тестовой неделе таких было **254**. Поэтому качество по этой метрике нельзя считать оценкой для всей аудитории.

**Всего одна тестовая неделя.** Итоговые числа относятся к 16–22 августа 2021 года. На другом периоде порядок моделей может измениться. К тому же мы несколько раз выбирали новые идеи, глядя на одну и ту же валидацию. Это повышает риск случайно подстроиться под неё.

**Доверительные интервалы.** Bootstrap помогает увидеть разброс результатов между пользователями в выбранной неделе. Но он не учитывает, как меняется качество между неделями или после нового обучения модели. Поэтому небольшой прирост NDCG сам по себе ещё не доказывает, что метод в целом лучше.

Полный список экспериментов и неудачных вариантов, включая спектральный отбор, остался в `research_log.ipynb`. Здесь собрана основная последовательность, которую проще запускать и проверять. Для повторения нужны CSV KION и зависимости из `requirements.txt`. Ниже выводятся версии библиотек и число деревьев обученных моделей.


In [ ]:
import sklearn
print('Python:',sys.version.split()[0], 'NumPy:',np.__version__,
      'pandas:',pd.__version__, 'CatBoost:',catboost.__version__,
      'sklearn:',sklearn.__version__)
print('Количество деревьев:',{name:m.tree_count_ for name,m in trained_models.items()})

## Итог

На тестовой неделе лучший NDCG@10 среди основных моделей получился у **Exponential: 0.107744**. У **Manual+** было **0.104645**, а у **Fixed Windows** с той же ручной основой **0.106191**. Добавление маршрутизации по активности почти ничего не изменило: **0.107710**.

Однако разница между Exponential и Manual+ небольшая, а 95%-й bootstrap-интервал для неё **от −0.005649 до 0.012274** включает ноль. Поэтому честный вывод такой: на этом тесте экспоненциальная память показала лучший результат, но данных недостаточно, чтобы утверждать, что она стабильно превосходит ручные признаки.

Для меня главный результат работы в том, что удалось на одинаковых кандидатах сравнить несколько способов учитывать прошлые действия пользователя и увидеть, как результаты на валидации расходятся с итоговым тестом. Дальше имеет смысл проверять их на нескольких независимых неделях и на более широкой выборке пользователей.
